In [1]:
import csv
import re
import time
from datetime import datetime
from typing import List, Dict

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

ROOT_URL = "https://www.amazon.in/gp/bestsellers/kitchen/3474656031"
OUTPUT_CSV = "amazon_top50.csv"
MAX_PRODUCTS = 50

# Evaluated once at the start of the script per instructions
# Format: YYYY-mm-dd HH:MM:SS
RUN_DATETIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Exact columns requested
FIELD_NAMES = [
    "scrape_datetime", "website_url", "title", "rank", "brand",
    "type", "type_2", "tonnage", "star_rating", "selling_price",
    "review count", "review_rating"
]

def make_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,2200")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip())

def extract_asin(url: str) -> str:
    m = re.search(r"/dp/([A-Z0-9]{10})", url)
    return m.group(1) if m else ""

def get_product_links(driver: webdriver.Chrome, root_url: str, max_products: int) -> List[str]:
    links: List[str] = []
    seen_asins = set()
    page = 1

    # Bestsellers span multiple pages (30 per page); loop to get 50.
    while len(links) < max_products and page <= 2:
        current_url = root_url if page == 1 else f"{root_url}/ref=zg_bs_pg_{page}_kitchen?ie=UTF8&pg={page}"
        driver.get(current_url)
        
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a.a-link-normal[href*='/dp/']"))
            )
        except TimeoutException:
            print(f"Timeout waiting for elements on page {page}")
            break

        time.sleep(3) # Allow dynamic widgets/images to render

        candidates = driver.find_elements(By.CSS_SELECTOR, "a.a-link-normal[href*='/dp/']")

        for a in candidates:
            href = (a.get_attribute("href") or "").split("?")[0]
            asin = extract_asin(href)
            if not asin or asin in seen_asins:
                continue
            seen_asins.add(asin)
            links.append(href)
            if len(links) >= max_products:
                break
        page += 1

    return links

def parse_title_attributes(title: str):
    """Extracts granular characteristics directly from product title string"""
    # Brand is typically the first word 
    brand = title.split(' ')[0] if title else ""
    
    # Inverter/Non-inverter
    type_1 = "Inverter" if "inverter" in title.lower() else "Non-Inverter"
    
    # Split/Window
    if "split" in title.lower():
        type_2 = "Split"
    elif "window" in title.lower():
        type_2 = "Window"
    else:
        type_2 = ""
        
    # Tonnage Regex (e.g., '1.5' from '1.5 Ton')
    tonnage_match = re.search(r'(\d+(?:\.\d+)?)\s*[Tt]on', title)
    tonnage = tonnage_match.group(1) if tonnage_match else ""
    
    # Star Rating Regex (e.g., '5' from '5 Star')
    star_match = re.search(r'(\d)\s*[Ss]tar', title)
    star_rating = star_match.group(1) if star_match else ""
    
    return brand, type_1, type_2, tonnage, star_rating

def first_text(driver: webdriver.Chrome, selectors: List[Dict[str, str]]) -> str:
    for sel in selectors:
        try:
            el = driver.find_element(sel["by"], sel["value"])
            txt = clean_text(el.text)
            if txt:
                return txt
        except NoSuchElementException:
            continue
    return ""

def scrape_product(driver: webdriver.Chrome, url: str, rank: int) -> dict:
    driver.get(url)
    time.sleep(2)
    
    title = first_text(driver, [{"by": By.ID, "value": "productTitle"}])
    brand, type_1, type_2, tonnage, star_rating = parse_title_attributes(title)
    
    price_text = first_text(driver, [
        {"by": By.CSS_SELECTOR, "value": "span.a-price.aok-align-center span.a-offscreen"},
        {"by": By.CSS_SELECTOR, "value": "span.a-price span.a-offscreen"},
        {"by": By.ID, "value": "corePriceDisplay_desktop_feature_div"}
    ])
    # Extract only the integer representation for Price
    selling_price = re.sub(r'[^\d]', '', price_text.split('.')[0]) if price_text else ""
    
    rating_text = first_text(driver, [
        {"by": By.CSS_SELECTOR, "value": "span[data-hook='rating-out-of-text']"},
        {"by": By.CSS_SELECTOR, "value": "i.a-icon-star span.a-icon-alt"}
    ])
    # Match the X.X rating explicitly
    review_rating_match = re.search(r'(\d\.\d)', rating_text)
    review_rating = review_rating_match.group(1) if review_rating_match else ""
    
    rev_count_text = first_text(driver, [{"by": By.ID, "value": "acrCustomerReviewText"}])
    review_count = re.sub(r'[^\d]', '', rev_count_text)

    return {
        "scrape_datetime": RUN_DATETIME,
        "website_url": url,
        "title": title,
        "rank": str(rank),
        "brand": brand,
        "type": type_1,
        "type_2": type_2,
        "tonnage": tonnage,
        "star_rating": star_rating,
        "selling_price": selling_price,
        "review count": review_count,
        "review_rating": review_rating
    }

def save_csv(rows: List[dict], out_file: str) -> None:
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        # Instructs csv.writer to explicitly add "" quotes around ALL items 
        writer = csv.DictWriter(f, fieldnames=FIELD_NAMES, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

def main() -> None:
    driver = make_driver()
    rows: List[dict] = []
    try:
        links = get_product_links(driver, ROOT_URL, MAX_PRODUCTS)
        if len(links) < MAX_PRODUCTS:
            print(f"Warning: found only {len(links)} product links from root page.")

        for idx, link in enumerate(links[:MAX_PRODUCTS], start=1):
            try:
                row = scrape_product(driver, link, idx)
                rows.append(row)
                print(f"[{idx}/{MAX_PRODUCTS}] OK")
            except Exception as ex:
                print(f"[{idx}/{MAX_PRODUCTS}] FAILED: {link} | {ex}")
                # Append fallback row preserving the ranking & URL limits
                rows.append({
                    "scrape_datetime": RUN_DATETIME,
                    "website_url": link,
                    "title": "",
                    "rank": str(idx),
                    "brand": "",
                    "type": "",
                    "type_2": "",
                    "tonnage": "",
                    "star_rating": "",
                    "selling_price": "",
                    "review count": "",
                    "review_rating": ""
                })

        save_csv(rows, OUTPUT_CSV)
        print(f"Saved {len(rows)} rows to {OUTPUT_CSV}")

    finally:
        driver.quit()

if __name__ == "__main__":
    main()

[1/50] OK
[2/50] OK
[3/50] OK
[4/50] OK
[5/50] OK
[6/50] OK
[7/50] OK
[8/50] OK
[9/50] OK
[10/50] OK
[11/50] OK
[12/50] OK
[13/50] OK
[14/50] OK
[15/50] OK
[16/50] OK
[17/50] OK
[18/50] OK
[19/50] OK
[20/50] OK
[21/50] OK
[22/50] OK
[23/50] OK
[24/50] OK
[25/50] OK
[26/50] OK
[27/50] OK
[28/50] OK
[29/50] OK
[30/50] OK
[31/50] OK
[32/50] OK
[33/50] OK
[34/50] OK
[35/50] OK
[36/50] OK
[37/50] OK
[38/50] OK
[39/50] OK
[40/50] OK
[41/50] OK
[42/50] OK
[43/50] OK
[44/50] OK
[45/50] OK
[46/50] OK
[47/50] OK
[48/50] OK
[49/50] OK
[50/50] OK
Saved 50 rows to amazon_top50.csv
